# Hybrid Attack Walkthrough

This notebook explains what the Hybrid attack is doing on top of DAGER.

Pipeline summary:

1. run the DAGER front-end to recover L1 token candidates
2. optionally initialize from DAGER decoder candidates
3. optimize continuous token embeddings against gradient-matching loss
4. optionally add an LM prior inside the candidate set
5. project the continuous embeddings back to candidate tokens

This notebook traces a few optimization steps rather than running a full benchmark.

In [ ]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
from notebooks.attack_walkthrough_helpers import (
    SMALL_SENTENCES,
    LONG_SENTENCES,
    build_attack_args,
    make_labels,
    trace_hybrid_attack,
)

pd.set_option('display.max_colwidth', 200)

## Small Sentences

In [ ]:
small_sentences = SMALL_SENTENCES
small_args = build_attack_args(
    dataset='sst2',
    batch_size=len(small_sentences),
    model_path='gpt2',
    extra_args=[
        '--rank_tol', '1e-8',
        '--max_ids', '64',
        '--n_steps', '8',
        '--hybrid_init_mode', 'dager',
        '--hybrid_projection_mode', 'candidate_final',
    ],
)
small_labels = make_labels(len(small_sentences), small_args.device, label=0)
small_trace = trace_hybrid_attack(small_args, small_sentences, labels=small_labels, traced_steps=8)

print('Rank:', small_trace['rank'])
small_trace['reference_table']

In [ ]:
small_trace['l1_trace']['candidate_counts']

In [ ]:
small_trace['init_table']

In [ ]:
small_trace['step_history']

## Long Sentences

In [ ]:
long_sentences = LONG_SENTENCES
long_args = build_attack_args(
    dataset='sst2',
    batch_size=len(long_sentences),
    model_path='gpt2',
    extra_args=[
        '--rank_tol', '1e-8',
        '--max_ids', '64',
        '--n_steps', '8',
        '--hybrid_init_mode', 'dager',
        '--hybrid_projection_mode', 'candidate_final',
    ],
)
long_labels = make_labels(len(long_sentences), long_args.device, label=0)
long_trace = trace_hybrid_attack(long_args, long_sentences, labels=long_labels, traced_steps=8)

print('Rank:', long_trace['rank'])
long_trace['reference_table']

In [ ]:
long_trace['l1_trace']['candidate_counts'].head(40)

In [ ]:
long_trace['init_table']

In [ ]:
long_trace['step_history']

## Why Higher Batch Sizes Usually Hurt

Things to keep in mind while reading the trace:

- Larger batches increase the effective rank and make the L1 candidate sets much noisier.
- If the correct token never survives DAGER's L1 stage, Hybrid cannot recover it later when projection stays inside the candidate set.
- Under small batches, DAGER is often already perfect, so Hybrid mostly adds time rather than quality.
- The more useful stress tests are usually around batch sizes `32`, `64`, and `128`, not the perfect low-batch regime.